# modular-diffusion: who writes the addresses of a picture, and how often they are wrong

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.11.1/colab/layout_write.ipynb)

The parts of a modular picture model are served by declared addresses: which things a picture holds, how many, which color on which, and where each goes on a 3 x 3 grid. A wrong address serves the wrong part at full weight, so the writer's error rate is the system's leak. This notebook measures it (`geolip.modular_diffusion.layout`, `experiments.layout_write`); nothing is trained.

THE WRITERS: the text parser of `thought-image-seeds` alone (no language model); Qwen3.5-2B at a pinned revision; and its LoRA fine-tune `AbstractPhil/qwen3.5-2b-think-symbolic-image` at the tag `lucia-v1.2`, merged into the weights. THE PROMPTS: 600 prompts whose answers are known by construction (100 each of count, absent, colour, place, relation and combined, from a seeded generator) and the first captions of COCO val2017 rows 0 to 99 (scored only for what needs no key). THE FORMS: a JSON list of boxes on the 0 to 1000 grid (the model family's own box format), one line a thing `name | color | column | row`, and the fine-tune's own text-only form (a written chain of thought, from the parser's records or from the description itself), each box and line form with six fixed worked examples and with none. A thirteenth cell asks the fine-tune its trained whole-row form (the chain, the wordings and its question lines, which are read as records too). 13 cells; one user turn and no system turn; greedy decoding, thinking off, at most 512 new tokens, half precision.

1. install at release tags and print the environment;
2. the two models at their pins; the adapter card's convention for the system turn; the 100 real captions; the scorer's own controls on fake writers (its own layouts score 1; broken answers fail with their own kinds); every cell on 8 prompts a family with its seconds; the projected minutes; the smoke shipped;
3. the 13 cells, one at a time; each finished cell folder (its raw answers, its scored rows, its summary and controls) is shipped at once; the prompt-by-prompt comparisons; a progress line a minute; the summary; the machine is given back only when every finished job and the report were shipped.

RESUME: run the cells again. Finished jobs keep their markers under `OUT`. On a new machine, finished jobs are fetched back from the Hub with their files, so nothing runs twice.

In [ ]:
# C1 - install at tags (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[layout,hub,data] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.11.1"
%pip install -q "thought-image-seeds @ git+https://github.com/AbstractEyes/thought-image-seeds@v0.4.0"
# without its dependencies: its own pin of thought-image-seeds is older than the parser measured here
%pip install -q --no-deps "qwen-symbolic-thinking @ git+https://github.com/AbstractEyes/qwen-symbolic-thinking@v0.1.71"
import importlib.metadata, subprocess, sys
try:                 # an old torchao beside PEFT stops PEFT from building a LoRA (PEFT wants 0.16 or newer, or none)
    _tao = importlib.metadata.version("torchao")
    if tuple(int(x) for x in _tao.split(".")[:2]) < (0, 16):
        subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
        print("torchao", _tao, "removed: PEFT cannot build a LoRA beside it")
except (importlib.metadata.PackageNotFoundError, ValueError):
    pass
import os, sys, time, json, gc, hashlib, shutil, threading
import torch, geolip.modular_diffusion as MD
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 11, 1), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
import transformers, huggingface_hub, thought_seeds, qwen_symbolic_thinking
assert thought_seeds.__version__ == "0.4.0" and qwen_symbolic_thinking.__version__ == "0.1.71", \
    (thought_seeds.__version__, qwen_symbolic_thinking.__version__)
from thought_seeds.textspec import spec_from_text
from qwen_symbolic_thinking.prompts import spec_short
try:
    import peft
    PEFT = peft.__version__
except ImportError:
    PEFT = "not installed (the plain merge is used)"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| huggingface_hub", huggingface_hub.__version__,
      "| peft", PEFT, "| thought-image-seeds", thought_seeds.__version__, "| qwen-symbolic-thinking",
      qwen_symbolic_thinking.__version__, "| package", MD.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("card:", props.name, f"| memory {props.total_memory / 2**30:.1f} GiB | cpus: {os.cpu_count()}")
else:
    print("card: NONE (choose a GPU runtime)")
OUT = "/content/layout_write"       # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

The base model comes from the Hub at its pinned revision and the adapter at its tag (the tag's commit is checked). No cell has a system turn (the adapter's model card says it was trained without one); the card's lines about the system turn are printed, and the cell stops if the card no longer says "no system prompt". The real prompts are the first captions of rows 0 to 99 of the first COCO val2017 parquet file, at the revision the capture notebooks use. Then the scorer's own controls run on fake writers: the generator's layouts written in the box and the line forms must score 1 on every prompt, and every deliberately broken answer must fail with its own kind. Then every one of the 13 cells runs on 8 prompts of each family and 8 real captions on this card, with its seconds and its repeat check; the whole run is projected from those seconds; the smoke folder is shipped.

In [ ]:
# C2 - the models, the captions, the controls, the smoke, the projection
import re
from huggingface_hub import snapshot_download, hf_hub_download, HfApi
import pyarrow.parquet as pq
from geolip.modular_diffusion import hub, run, layout
from geolip.modular_diffusion.experiments import layout_write
BASE, LUCIA, RUNS, SMOKE = f"{OUT}/base", f"{OUT}/lucia", f"{OUT}/runs", f"{OUT}/smoke"
REAL = f"{OUT}/real_captions.json"
REPO = "AbstractPhil/geolip-multirotary-diffusion-collective"   # the results repository ("" = nothing shipped)
PREFIX = "modular_diffusion/conditioning/address_writer"
NOT_SHIPPED = ("SHIPPED", "FROM_HUB")
os.makedirs(RUNS, exist_ok=True)

# the models and the captions
api = HfApi()
assert api.model_info("Qwen/Qwen3.5-2B", revision="15852e8c16360a2fea060d615a32b45270f8a8fc").sha == "15852e8c16360a2fea060d615a32b45270f8a8fc", "the base model's revision"
lucia_sha = api.model_info("AbstractPhil/qwen3.5-2b-think-symbolic-image", revision="lucia-v1.2").sha
print("adapter tag lucia-v1.2 ->", lucia_sha)
assert lucia_sha.startswith("9366fbbe"), "the adapter's tag does not point at the expected commit"
snapshot_download(repo_id="Qwen/Qwen3.5-2B", revision="15852e8c16360a2fea060d615a32b45270f8a8fc", local_dir=BASE)
snapshot_download(repo_id="AbstractPhil/qwen3.5-2b-think-symbolic-image", revision=lucia_sha, local_dir=LUCIA,
                  allow_patterns=["adapter_config.json", "adapter_model.safetensors", "README.md"])
assert all(os.path.exists(f"{LUCIA}/{f}") for f in ("adapter_config.json", "adapter_model.safetensors", "README.md")), \
    "the adapter's files are not at the root of its repository: look at its file list"
card = open(f"{LUCIA}/README.md", encoding="utf-8").read()
said = [ln.strip() for ln in card.splitlines() if "system" in ln.lower()]
print("the adapter card on the system turn:" + ("".join("\n  " + ln for ln in said) if said else " (no line names it)"))
LUCIA_SYSTEM = ""        # no system turn for every chain cell: the card's own convention ("no system prompt")
assert any(re.search(r"(?i)\bno system prompt\b", ln) for ln in said), (
    "the adapter card no longer says 'no system prompt': read the lines above before running the chain cells")
print("the chain cells' system turn:", repr(LUCIA_SYSTEM) if LUCIA_SYSTEM else "none")
PARQUET = hf_hub_download(repo_id="lmms-lab-encoder/COCO-Caption2017", repo_type="dataset", revision="3bdd5827e243cc3084ac69a1111e69c3ab9193ff",
                          filename="data/val-00000-of-00002.parquet", local_dir=f"{OUT}/coco")
captions = []      # the first caption of rows 0 to 99, read without the pictures (the rule of the capture notebooks)
for b in pq.ParquetFile(PARQUET).iter_batches(batch_size=128, columns=["answer"]):
    for a in b.column(0).to_pylist():
        captions.append("" if a is None else str((a[0] if a else "") if isinstance(a, (list, tuple)) else a))
    if len(captions) >= 100:
        break
captions = captions[:100]
assert len(captions) == 100 and all(c.strip() for c in captions), "the real captions"
json.dump(captions, open(REAL, "w", encoding="utf-8"), indent=1)
print("real captions:", len(captions), "| the first:", captions[0])

# the cells: (tag, arguments); no cell has a system turn
DEVICE = "cuda"
DTYPE = "bf16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "fp16"     # half precision
BATCH = 8
WRITERS = {"base": f"writer={BASE}", "lucia12": f"writer={BASE} adapter={LUCIA}"}
SYSTEM_ARG = f"system={LUCIA_SYSTEM}"
COMMON = f"device={DEVICE} dtype={DTYPE} batch={BATCH} max_new_tokens=512 local_only=1 seed=0 threads=4"
CELLS = [("parser_spec_ex0", "writer=parser form=spec examples=0")]
for w in ("base", "lucia12"):
    for form in ("box", "cell"):
        for ex in (1, 0):
            CELLS.append((layout_write.cell_name(w, form, ex), f"{WRITERS[w]} form={form} examples={ex}"))
CELLS.append(("lucia12_chain_records_ex0", f"{WRITERS['lucia12']} form=chain_records examples=0 {SYSTEM_ARG}"))
CELLS.append(("lucia12_chain_text_ex0", f"{WRITERS['lucia12']} form=chain_text examples=0 {SYSTEM_ARG}"))
CELLS.append(("lucia12_chain_full_records_ex0", f"{WRITERS['lucia12']} form=chain_full_records examples=0 {SYSTEM_ARG}"))
CELLS.append(("base_chain_text_ex0", f"{WRITERS['base']} form=chain_text examples=0 {SYSTEM_ARG}"))
# the comparisons: per form and examples setting the fine-tune against the base; per model box against cell with examples
COMPARES = [(f"cmp_lucia12_vs_base_{f}", f"lucia12_{f}", f"base_{f}") for f in ("box_ex1", "box_ex0", "cell_ex1", "cell_ex0",
                                                                               "chain_text_ex0")]
COMPARES += [(f"cmp_{w}_box_vs_cell_ex1", f"{w}_box_ex1", f"{w}_cell_ex1") for w in ("base", "lucia12")]
assert len(CELLS) == 13 and len({t for t, _ in CELLS}) == 13 and len(COMPARES) == 7

def job_list(runs, extra=""):
    jobs = [(t, f"variant=write {a} {COMMON} real={REAL} repeat_n=20{extra}", [], "cell") for t, a in CELLS]
    jobs += [(t, f"variant=write_compare a={runs}/{x} b={runs}/{y}", [x, y], "compare") for t, x, y in COMPARES]
    return jobs

# the smoke: the scorer's own controls on fake writers (all 600 prompts), then every cell on 8 prompts a family
shutil.rmtree(SMOKE, ignore_errors=True)
S = {}
for form in ("box", "cell"):
    S[form] = run.main("layout_write", f"variant=write writer=fake:truth form={form} examples=1 repeat_n=20".split()
                       + [f"out={SMOKE}/fake_truth_{form}_ex1"])
    fam = S[form]["summary"]["families"]
    assert all(fam[f]["scores"]["address_right"]["mean"] == 1.0 for f in layout.FAMILIES), f"control (a) in the {form} form"
    assert S[form]["controls"]["a"]["holds"] is True, f"control (a) in the {form} form"
    assert S[form]["controls"]["c"]["holds"] is True, f"control (c) in the {form} form"
    print(f"control (a) and (c), {form} form: the generator's layouts score 1 on all 600 prompts; every broken answer fails with its kind")
SMOKE_N = 8
SEC = {}
for t, a in CELLS:
    t0 = time.time()
    r = run.main("layout_write", f"variant=write {a} {COMMON} real={REAL} first_n={SMOKE_N} repeat_n={BATCH}".split()
                 + [f"out={SMOKE}/{t}"])
    SEC[t] = {"seconds": time.time() - t0, "loading": r["seconds_loading"], "writing": r["seconds_writing"],
              "answers": r["prompts"] + (r["controls"]["d"].get("asked") or 0)}
    print(f"{t}: {r['prompts']} prompts in {SEC[t]['seconds']:.0f} s (loading {r['seconds_loading']:.0f} s); parsed "
          f"{r['summary']['all']['scores']['parsed']['mean']:.2f}; address right {r['summary']['all']['scores']['address_right']['mean']:.2f}; "
          f"repeat {r['controls']['d'].get('same')}/{r['controls']['d'].get('compared')}", flush=True)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# the projection: each cell's seconds an answer (its writing plus the repeat, over the answers it gave) times the whole
# cell's answers (700 prompts and 24 asked again), plus its loading
FULL_ANSWERS = 600 + 100 + -(-20 // BATCH) * BATCH
COST = {}
for t, _ in CELLS:
    per = (SEC[t]["seconds"] - SEC[t]["loading"]) / max(1, SEC[t]["answers"])
    COST[t] = SEC[t]["loading"] + 30 + per * FULL_ANSWERS
for t, _, _ in COMPARES:
    COST[t] = 15.0
JOBS = job_list(RUNS)
INSTRUMENTS = {"smoke_seconds": SEC, "projected_seconds": COST, "card": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
               "dtype": DTYPE, "batch": BATCH, "lucia_commit": lucia_sha, "lucia_system": LUCIA_SYSTEM, "card_system_lines": said,
               "real_captions": len(captions), "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
               "peft": PEFT, "thought_seeds": thought_seeds.__version__, "qwen_symbolic_thinking": qwen_symbolic_thinking.__version__,
               "package": MD.__version__}}
json.dump(INSTRUMENTS, open(f"{SMOKE}/instruments.json", "w", encoding="utf-8"), indent=1)
print(f"the job list: {len(JOBS)} jobs ({len(CELLS)} cells, {len(COMPARES)} comparisons), about {sum(COST.values()) / 60:.0f} projected "
      f"minutes on this card with one runner")
if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")          # placed in this process's environment only
    assert hub.ship(SMOKE, REPO, f"{PREFIX}/_smoke", "address writer: smoke run", ignore=NOT_SHIPPED), (
        "the smoke folder was not shipped: check the token and the repository before the long run")

One runner works the job list in order: the 13 cells, then the 7 prompt-by-prompt comparisons (each waits for its two cells). Each finished job folder is shipped at once to `REPO` (a cell with its raw answers `outputs.jsonl`, its scored rows `rows.jsonl`, its `result.json` and `result.md`), with the token from the notebook's secret `HF_TOKEN` (placed in this process's environment only; the jobs themselves run without it). At the end the summary, the roster, its log and the instruments are shipped to `_report`. The machine is given back (`RELEASE`) only when the report and every finished job were shipped; if an upload failed, the machine is kept and the cell names the jobs.

In [ ]:
# C3 - the job list, each finished job shipped, a progress line a minute, the summary, the machine
from geolip.modular_diffusion import jobs
RELEASE = True                     # give the machine back at the end (only when every finished job and the report were shipped)
EXTRA = ""                         # more arguments for every cell (empty in a real run)
PROGRESS_S = 60
JOBS = job_list(RUNS, EXTRA)
kind = {t: k for t, _, _, k in JOBS}
needs = {t: n for t, _, n, _ in JOBS}
tags = [t for t, *_ in JOBS]
roster = f"{OUT}/roster.txt"
cap = lambda t: max(30, int(3 * COST[t] / 60))
with open(roster, "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(f"{t} | {cap(t)} | geolip.modular_diffusion.run | layout_write {a}" + (f" | needs={','.join(n)}" if n else "")
                       for t, a, n, _ in JOBS) + "\nEND\n")
mark = lambda t, m: os.path.exists(f"{RUNS}/{t}/{m}")
if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
CELL_FILES = ("outputs.jsonl", "rows.jsonl", "result.json")

def fetch_back():
    # on a new machine: finished jobs come back with their files, so nothing runs twice; a job comes back only when what
    # it reads came back too
    try:
        got = snapshot_download(repo_id=REPO, allow_patterns=[f"{PREFIX}/{t}/*" for t in tags], local_dir=f"{OUT}/hub_back")
    except Exception as e:
        print("nothing fetched back from the Hub:", repr(e)[:200])
        return
    for t in tags:
        src, dst = f"{got}/{PREFIX}/{t}", f"{RUNS}/{t}"
        if mark(t, "DONE") or not os.path.exists(f"{src}/result.json"):
            continue
        if kind[t] == "cell" and not all(os.path.exists(f"{src}/{f}") for f in CELL_FILES):
            continue                                             # its answers are not on the Hub: it is run again
        if not all(mark(n, "DONE") for n in needs[t]):
            continue
        shutil.copytree(src, dst, dirs_exist_ok=True)
        open(f"{dst}/DONE", "w").write("fetched back from the Hub\n")
        open(f"{dst}/FROM_HUB", "w").write("fetched back from the Hub\n")

if REPO:
    fetch_back()
for t in tags:      # a claim left by an interrupted session would hold its job forever
    if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED")):
        os.remove(f"{RUNS}/{t}/CLAIM")
def after_job(tag, folder, ok):
    if ok and REPO:
        if hub.ship(folder, REPO, f"{PREFIX}/{tag}", f"address writer: {tag}", ignore=NOT_SHIPPED):
            open(f"{folder}/SHIPPED", "w").write("shipped to the Hub\n")      # kept across sessions on this machine
        else:
            print(f"{tag}: NOT SHIPPED (its folder stays on this machine: {folder})", flush=True)

shipped_or_fetched = lambda t: mark(t, "SHIPPED") or mark(t, "FROM_HUB")
for t in tags:      # a job that finished in an earlier session but whose upload failed: shipped again now
    if REPO and mark(t, "DONE") and not shipped_or_fetched(t):
        after_job(t, f"{RUNS}/{t}", True)

t0 = time.time()
left0 = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
print(f"start: {sum(mark(t, 'DONE') for t in tags)}/{len(tags)} jobs already done; about {left0 / 60:.0f} min to run; one runner")
runner = threading.Thread(target=jobs.run_roster, daemon=True,
                          kwargs=dict(roster=roster, out=RUNS, stop=f"{OUT}/STOP", log=f"{OUT}/roster.log", card_room_mib=None,
                                      ram_free_gb=4.0, max_hours=20.0, claim=True, after_job=after_job, wait_s=5.0,
                                      env={"OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4", "HF_HUB_OFFLINE": "1"}))
runner.start()
while runner.is_alive():
    time.sleep(PROGRESS_S)
    done, failed = sum(mark(t, "DONE") for t in tags), sum(mark(t, "FAILED") for t in tags)
    left = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
    spent = (time.time() - t0) / 60
    rate = (left0 - left) / 60 / spent if left0 > left and spent > 0 else 1.0     # projected minutes done a minute
    print(f"jobs done {done}/{len(tags)} (failed {failed}); {spent:.0f} min spent; about {left / 60 / rate:.0f} min left", flush=True)

summary = layout_write.summarize(RUNS)
print(summary)
REPORT = f"{OUT}/report"
os.makedirs(REPORT, exist_ok=True)
open(f"{REPORT}/summary.md", "w", encoding="utf-8").write(summary)
json.dump(INSTRUMENTS, open(f"{REPORT}/instruments.json", "w", encoding="utf-8"), indent=1)
for f in ("roster.txt", "roster.log"):
    if os.path.exists(f"{OUT}/{f}"):
        shutil.copy(f"{OUT}/{f}", f"{REPORT}/{os.path.basename(f).replace('.log', '_log.txt')}")
for t in tags:      # the logs of the jobs that failed
    if mark(t, "FAILED"):
        for f in ("run.log", "log.txt"):
            if os.path.exists(f"{RUNS}/{t}/{f}"):
                shutil.copy(f"{RUNS}/{t}/{f}", f"{REPORT}/failed_{t}_{f.replace('.', '_')}.txt")
shipped = hub.ship(REPORT, REPO, f"{PREFIX}/_report", "address writer: summary, roster and instruments") if REPO else None
unshipped = [t for t in tags if mark(t, "DONE") and not shipped_or_fetched(t)]
print("report shipped:", shipped, "| finished jobs not on the Hub:", ", ".join(unshipped) or "none")
if RELEASE and shipped and not unshipped:
    from google.colab import runtime
    runtime.unassign()
else:
    print("THE MACHINE IS KEPT:", "the report was not shipped" if not shipped else
          f"these finished jobs were not shipped: {', '.join(unshipped)} (run C3 again to retry)" if unshipped else "RELEASE is False")